In [1]:
import numpy as np
import pandas as pd
import time
import json
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

In [3]:
# 1. Загрузка меток и временных статистик
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')
y_all = np.concatenate([y_train, y_test])

with open('../data/processed/vectorization_time.json', 'r') as f:
    v_times = json.load(f)


In [4]:
def run_clustering_experiment(name, method_key):
    print(f"\n Эксперимент: {name}")
    
    # 1. Загрузка векторов
    # Объединяем train и test, чтобы кластеризовать весь датасет
    X_train = np.load(f'../data/processed/train_{method_key}_vectors.npy')
    X_test = np.load(f'../data/processed/test_{method_key}_vectors.npy')
    X_all = np.vstack([X_train, X_test])
    
    start_model_time = time.time()
    
    # 2. Предобработка (Нормализация обязательна для KMeans)
    X_norm = normalize(X_all)

    # 3. Снижение размерности (PCA) 
    # Для разреженных методов (TF-IDF) это критично, для BERT опционально, но оставим для честности сравнения
    pca = PCA(n_components=50, random_state=42)
    X_reduced = pca.fit_transform(X_norm)

    # 4. Кластеризация KMeans
    kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_reduced)

    # 5. Расчет метрик
    # Silhouette считаем на уменьшенных данных для скорости
    sil = silhouette_score(X_reduced, clusters, sample_size=2000, random_state=42)
    ari = adjusted_rand_score(y_all, clusters)
    
    model_duration = round(time.time() - start_model_time, 2)
    vec_time = v_times[method_key]
    total_time = round(vec_time + model_duration, 2)
    
    print(f"Silhouette: {sil:.4f}")
    print(f"ARI: {ari:.4f}")
    print(f"Model Time: {model_duration}s (Vec Time: {vec_time}s)")

    return {
        "Method": name,
        "Silhouette": sil,
        "ARI": ari,
        "Vectorization Time": vec_time,
        "Model Training Time": model_duration,
        "Total Time": total_time
    }

In [5]:
# Список методов для сопоставления с именами файлов
methods_to_compare = [
    ("Binary", "binary"),
    ("Bag of Words", "bow"),
    ("TF-IDF Standard", "tfidf_std"),
    ("TF-IDF Bigrams", "tfidf_bigrams"),
    ("Word2Vec", "w2v"),
    ("Doc2Vec", "d2v"),
    ("BERT", "bert")
]

results = []


In [6]:
for public_name, file_key in methods_to_compare:
    res = run_clustering_experiment(public_name, file_key)
    results.append(res)


 Эксперимент: Binary
Silhouette: 0.0633
ARI: 0.0362
Model Time: 18.96s (Vec Time: 1.78s)

 Эксперимент: Bag of Words
Silhouette: 0.0840
ARI: 0.0249
Model Time: 16.15s (Vec Time: 1.92s)

 Эксперимент: TF-IDF Standard
Silhouette: 0.0820
ARI: 0.0775
Model Time: 16.19s (Vec Time: 1.64s)

 Эксперимент: TF-IDF Bigrams
Silhouette: 0.0766
ARI: 0.0796
Model Time: 16.33s (Vec Time: 4.93s)

 Эксперимент: Word2Vec
Silhouette: 0.0529
ARI: 0.0554
Model Time: 1.39s (Vec Time: 11.2s)

 Эксперимент: Doc2Vec
Silhouette: 0.0152
ARI: 0.1593
Model Time: 1.67s (Vec Time: 78.44s)

 Эксперимент: BERT
Silhouette: 0.1007
ARI: 0.3748
Model Time: 4.33s (Vec Time: 568.09s)


In [7]:
# 6. Итоговая таблица
results_df = pd.DataFrame(results).sort_values(by="ARI", ascending=False)

print("\nИтоговая сравнительная таблица кластеризации")
print(results_df)

import os
os.makedirs("../results/tables/", exist_ok=True)

# Сохраняем расширенные результаты
results_df.to_csv("../results/tables/clusterization_compare_compact.csv", index=False)


=== Итоговая сравнительная таблица кластеризации ===
            Method  Silhouette       ARI  Vectorization Time  \
6             BERT    0.100674  0.374839              568.09   
5          Doc2Vec    0.015158  0.159295               78.44   
3   TF-IDF Bigrams    0.076571  0.079649                4.93   
2  TF-IDF Standard    0.081981  0.077530                1.64   
4         Word2Vec    0.052943  0.055416               11.20   
0           Binary    0.063254  0.036222                1.78   
1     Bag of Words    0.084035  0.024872                1.92   

   Model Training Time  Total Time  
6                 4.33      572.42  
5                 1.67       80.11  
3                16.33       21.26  
2                16.19       17.83  
4                 1.39       12.59  
0                18.96       20.74  
1                16.15       18.07  
